# EP07｜資訊理論：Softmax、熵、交叉熵、KL 散度、困惑度

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chainchen1103/llm-from-scratch/blob/main/notebooks/ep07/ep07_information_theory.ipynb)

這份 notebook 對應網站上的 [EP07 圖文教學](https://chainchen1103.github.io/llm-from-scratch/episodes/ep07/)，段落和網站一樣，可以邊讀邊跑。EP06 的結論是**訓練語言模型＝最大概似估計**：讓模型給訓練資料裡每個「真正的下一個 token」的 log 機率加起來（叫 log 概似）越大越好；把這個總和取負號、再除以 token 數，就是 LLM 訓練時看到的損失。這留下三個問題：這個損失為什麼叫**交叉熵 cross-entropy**？**熵 entropy**、**KL 散度 KL divergence**、**困惑度 perplexity** 各量什麼？損失對分數的梯度長什麼樣子（EP05 把 Softmax 的 Jacobian 留到這集；EP06 玩具模型的梯度剛好是「實際次數 − 預期次數」）？答案都來自**資訊理論 information theory**：用「猜謎要問幾題」來量資訊。一個位置的損失是「看到正確答案時有多驚訝」，所有位置平均起來，正好是資料和模型的交叉熵。

1. 工具一：資訊量（含資訊量曲線的圖）
2. 工具二：熵
3. 工具三：交叉熵
4. 工具四：KL 散度（含三種猜法的堆疊長條圖）
5. 工具五：困惑度
6. **本集重點：為什麼 LLM 的損失是交叉熵**（熵與交叉熵的圖在第 8 節動手做畫）
7. Softmax 的導數：梯度為什麼是 q − y（含 Softmax Jacobian 的圖）
8. 動手做：從零寫出 `entropy`、`cross_entropy`、`kl_divergence`、`cross_entropy_loss`、`softmax_ce_grad`、`perplexity` 並用檢查函式測試；寫出畫熵與交叉熵的 `bernoulli_entropy_bits`、`draw_bernoulli`、`draw_ce_stack`；和 PyTorch 對照
9. 弄壞它：用 `np.log(softmax(z))` 算損失、用均方誤差當分類的損失（含比較圖）、KL 的順序放反、bit 和 nat 混用、熵裡的 0 × log 0 沒處理

最後是參考解答。

這份 notebook 只用到 NumPy 和 matplotlib，Colab 已經內建，不用安裝任何東西；前幾集寫好的 `softmax`（EP01）、`logsumexp`、`log_softmax`（EP02）、`numerical_gradient`（EP04）、`numerical_jacobian`（EP05）也已經定義在下面，不用匯入。在自己的電腦上，用 EP01 建立的 `llm-labs` 專案開啟即可。動手做的最後一個檢查、「和 PyTorch 對照」那一格，以及弄壞它 1、3 的一部分會和 PyTorch 比對：Colab 已經內建 PyTorch；沒裝 PyTorch 的話（EP01 步驟 7），那些地方會自動跳過。

**使用方式**：從上到下依序執行（Shift + Enter）。弄壞它的每一格，先在心裡猜結果再執行。參考解答在最後，先自己寫寫看。

In [ ]:
# Colab 已經內建 NumPy 和 matplotlib，不用安裝；在自己的電腦上，請用 EP01 建立的 llm-labs 環境。
# %pip install -q numpy matplotlib

import math
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| matplotlib", matplotlib.__version__)

這份 notebook 用 NumPy 2.5 測試過。程式格裡的註解是說明，印出來的結果就在格子下方，和網站上一樣。圖上的文字用英文，因為 matplotlib 預設的字型沒有中文，會變成方框。

下面是前幾集動手做寫好的函式，這集會用到：EP01 的 `softmax`；EP02 的 `logsumexp`（Softmax 的分母取 $\ln$）和 `log_softmax`（$z - \text{logsumexp}(z)$，直接算 log 機率、不會下溢）；EP04 的 `numerical_gradient`（每一格各做一次中央差分，$h$ 預設 `1e-5`）；EP05 的 `numerical_jacobian`（一欄一欄做中央差分）。最後的四個顏色和 `tidy` 是畫圖用的小工具：淡灰色格線、拿掉上面和右邊的框。

**寫法約定**：$p(x)$ 念「$x$ 的機率」，就是 EP06 的 $P(X = x)$；單獨寫 $p$、$q$ 時指整個分佈。熵、交叉熵、KL 散度的定義裡沒寫底數的 $\log$，用 $\log_2$ 或 $\ln$ 都可以，單位跟著底數走；損失和導數照 EP02、EP06，沒寫底數就是 $\ln$（`np.log`）。損失寫成 $\text{loss}$，和 EP06 的概似 $L(p)$ 分開。

In [ ]:
def softmax(z, axis=-1):                  # EP01
    """沿著 axis 把分數 z 轉成機率。z 可以是任意形狀。"""
    z = np.asarray(z, dtype=float)
    shifted = z - z.max(axis=axis, keepdims=True)   # 每一列減掉自己的最大值
    e = np.exp(shifted)
    return e / e.sum(axis=axis, keepdims=True)      # 每一列除以自己的總和


def logsumexp(z, axis=-1, keepdims=False):    # EP02
    """數值穩定地算 log(Σ exp(z))，沿著 axis 做。z 可以是任意形狀。
    回傳形狀和 z.sum(axis=axis, keepdims=keepdims) 一樣：
    z 形狀 (4, 7)、axis=-1 → (4,)；keepdims=True → (4, 1)。"""
    z = np.asarray(z, dtype=float)
    m = z.max(axis=axis, keepdims=True)                              # 每一列自己的最大值
    out = m + np.log(np.sum(np.exp(z - m), axis=axis, keepdims=True))
    if not keepdims:
        out = np.squeeze(out, axis=axis)                             # 拿掉長度 1 的那個維度
    return out


def log_softmax(z, axis=-1):              # EP02
    """回傳 log(softmax(z))，但不能先算 softmax 再取 log。輸出形狀和 z 一樣。"""
    z = np.asarray(z, dtype=float)
    return z - logsumexp(z, axis=axis, keepdims=True)                # keepdims=True 才能廣播回每一列


def numerical_gradient(f, x, h=1e-5):     # EP04
    """f 吃一個 NumPy 陣列、吐一個數。對 x 的每一格各做一次中央差分，
    回傳和 x 形狀一樣的陣列（每一格是 f 對那一格的偏導數）。不能改到呼叫端的 x。"""
    x = np.array(x, dtype=float)          # 複製一份、轉成小數：不改到呼叫端，整數輸入也能動一點點
    grad = np.zeros_like(x)
    for idx in np.ndindex(x.shape):       # (0,)、(1,)…；二維時是 (0, 0)、(0, 1)…
        old = x[idx]
        x[idx] = old + h
        f_plus = f(x)
        x[idx] = old - h
        f_minus = f(x)
        x[idx] = old                      # 復原：直接放回原值，不靠 +h、-h 抵消
        grad[idx] = (f_plus - f_minus) / (2 * h)
    return grad


def numerical_jacobian(f, x, h=1e-5):     # EP05
    """f 吃一維陣列、吐一維陣列。回傳形狀 (len(f(x)), len(x)) 的 Jacobian（中央差分，一欄一欄算）。
    不能改到呼叫端的 x。"""
    x = np.array(x, dtype=float)                  # 複製一份、轉成小數
    m = np.asarray(f(x)).size
    J = np.zeros((m, x.size))
    for j in range(x.size):
        old = x[j]
        x[j] = old + h
        f_plus = np.array(f(x), dtype=float)      # 複製：f 可能直接回傳 x 本身
        x[j] = old - h
        f_minus = np.array(f(x), dtype=float)
        x[j] = old                                # 放回原值
        J[:, j] = (f_plus - f_minus) / (2 * h)    # 第 j 欄：第 j 個輸入動一點點，每個輸出動多少
    return J


BLUE, RED, INK, MUTED = "#2B7FD6", "#E04E3E", "#1E2A44", "#5B6577"   # 畫圖用：藍、紅、文字、淡一點的文字


def tidy(ax, axis="both"):                # 畫圖用：淡灰色格線、拿掉上面和右邊的框
    ax.grid(True, axis=axis, color="0.88", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)

## 1. 工具一：資訊量

### 猜謎：要問幾個是非題

我心裡想一個 token，你只能問是非題。如果只有 8 個一樣可能的 token，最好的問法是每次砍一半：「是前 4 個嗎？」「是這 2 個嗎？」「是這個嗎？」，3 題一定猜中。$2^3 = 8$，所以題數就是 $\log_2 8 = 3$；$K$ 個一樣可能，就要 $\log_2 K$ 題。

EP00 的詞彙表有 151,936 個 token，$\log_2 151{,}936 \approx 17.21$。EP02 說要 18 個位元才夠替每個 token 編號，那是每個 token 編一個**一樣長**的號碼，位數只能是整數（17 位只有 131,072 種）。每個 token 一樣可能時，17.21 是平均位數的下限，可以是小數：把 100 個 token 排成的一串當成一個整體來編號，要 $100 \times 17.213 \approx 1{,}721.3$，進位成 1,722 位，平均每個 token 17.22 位。

### 資訊量 −log₂ p

$K$ 個一樣可能時，每一個的機率是 $p = \frac{1}{K}$，題數 $\log_2 K = \log_2 \frac{1}{p} = -\log_2 p$。推廣到任何機率：一件機率 $p(x)$ 的事 $x$ 發生，帶來

$$
I(x) = -\log_2 p(x)
$$

白話：機率取 $\log_2$ 再加負號，就是這件事發生時帶來多少資訊。

這叫**資訊量 information content**，也叫**驚訝程度 surprisal**。單位是**位元 bit**（EP02 的「一個 0 或 1」在這裡當單位）：公平硬幣丟出正面剛好 1 bit。

前五行是機率 $p$ 和它的資訊量（bit）；最後一行是「硬幣正面**而且**骰子 6 點」的資訊量，左邊用兩個機率相乘再算，右邊把兩個資訊量相加：

In [ ]:
for p in [0.5, 0.25, 0.125, 0.293, 0.01]:
    print(p, -np.log2(p))                        # 資訊量（bit）
print(-np.log2(1 / 2 * 1 / 6), -np.log2(1 / 2) + -np.log2(1 / 6))   # 硬幣正面「而且」骰子 6 點

- 機率越小，資訊量越大：1% 的事是 6.64 bit，發生了不太可能的事才「驚訝」。機率 1 的事是 0 bit（說了等於沒說）；機率到 0 是無限大（工具四）。EP00 的「冷」是 29.3%，出現時帶來 1.77 bit，不太驚訝。
- 最後一行：硬幣和骰子彼此獨立（EP06），「正面而且 6 點」的機率是 $\frac{1}{2} \times \frac{1}{6}$，資訊量剛好是 $1 + 2.585$。**獨立的事，機率相乘、資訊量相加**（EP02 的對數律），這是資訊量要用 log 的原因。

畫成曲線（紅點是上面的 0.5、「冷」0.293 和 0.01；圖上的 cold 是「冷」）：

In [ ]:
ps_curve = np.linspace(0.005, 1, 600)                # p = 0 時是無限大，從 0.005 開始畫
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(ps_curve, -np.log2(ps_curve), color=BLUE, linewidth=2)
marks = [(0.5, "fair coin: 1 bit", (8, 8)), (0.293, "'cold' 29.3%: 1.77 bit", (10, 6)),
         (0.01, "1%: 6.64 bit", (10, -4))]
for pm, text, off in marks:
    ax.plot([pm], [-np.log2(pm)], "o", color=RED, markersize=8)
    ax.annotate(text, (pm, -np.log2(pm)), xytext=off, textcoords="offset points", color=INK, fontsize=10)
ax.set(xlim=(0, 1.02), ylim=(0, 8), xlabel="probability p", ylabel="information -log2 p (bit)",
       title="surprisal: the less likely, the more information")
tidy(ax)
plt.show()

$p = 1$ 時是 0，$p$ 越小越高，往 0 靠近時一路往上衝。

### bit 和 nat

把 $\log_2$ 換成 $\ln$，單位叫 **nat**（natural 的縮寫）。用換底 $\log_2 x = \frac{\ln x}{\ln 2}$：

$$
1 \text{ nat} = \frac{1}{\ln 2} \approx 1.443 \text{ bit}, \qquad 1 \text{ bit} = \ln 2 \approx 0.693 \text{ nat}
$$

白話：兩者只差一個固定的倍數，就像公尺和公分。**PyTorch 和 LLM 的損失用 $\ln$，單位是 nat。**

第一行是「冷」的資訊量，左邊用 nat、右邊換成 bit；第二行是 151,936 個一樣可能時的資訊量，左邊 bit、右邊 nat：

In [ ]:
V = 151_936
print(-np.log(0.293), -np.log(0.293) / np.log(2))    # 「冷」的資訊量：nat，換成 bit
print(np.log2(V), np.log(V))                         # 151,936 個一樣可能：bit、nat

1.228 nat 就是 EP02、EP06 算過的 $\ln 0.293 = -1.228$ 去掉負號；換成 bit 和上一格只差最後一位（捨入誤差）。

## 2. 工具二：熵

### 熵是資訊量的期望值

如果每個 token 的機率不一樣呢？示範：下一個 token 只有 4 種可能，冷 $\frac{1}{2}$、熱 $\frac{1}{4}$、晴 $\frac{1}{8}$、暖 $\frac{1}{8}$。聰明的問法是先問最可能的：「是冷嗎？」一半的時候 1 題就結束；不是，再問「是熱嗎？」；還不是，問「是晴嗎？」就知道答案了。

印出每個 token 的資訊量、照這個問法平均要問幾題，以及資訊量用機率加權的平均：

In [ ]:
p4 = np.array([0.5, 0.25, 0.125, 0.125])     # 冷、熱、晴、暖（示範）
questions = np.array([1, 2, 3, 3])            # 照上面的問法，每個 token 要問到第幾題
print(-np.log2(p4), p4 @ questions, p4 @ -np.log2(p4))

每個 token 要問的題數，剛好就是它的資訊量 $-\log_2 p$；用機率加權平均（EP06 的期望值，`@` 是 EP03 的內積），平均 1.75 題，比 4 個一樣可能的 2 題少：常見的先問，平均就省。這個平均叫**熵 entropy**（念「商」）：

$$
H(p) = \sum_x p(x)\,\bigl(-\log p(x)\bigr)
$$

白話：**熵是資訊量的期望值，也就是平均的驚訝程度**。

$H(p)$ 念「$p$ 的熵」，用 $\log_2$ 是 bit，用 $\ln$ 是 nat。機率不是 $\frac{1}{2}$ 的次方時，題數（整數）不會剛好是 $-\log_2 p$；熵是平均題數的下限，很多次一起猜時可以很靠近它（資訊理論的創始人夏農 Shannon 1948 年證明，這裡不證）。

**$0 \times \log 0$ 當成 0**：機率 0 的事不會發生，不該算進平均（數學上 $p \ln p$ 在 $p$ 靠近 0 時也靠近 0）。但 `0 * np.log(0)` 是 `nan`，實作時要跳過 $p = 0$ 的項（弄壞它 5）。

### 均勻分佈的熵最大

`H_bits` 先假設 $p$ 裡沒有 0（弄壞它 5）。每一行是一個分佈（四捨五入到 3 位）和它的熵（bit）：

In [ ]:
def H_bits(p):
    p = np.asarray(p, dtype=float)
    return np.sum(p * -np.log2(p))           # 熵（bit）；先假設 p 裡沒有 0（弄壞它 5）

for p in [[0.5, 0.5], [0.9, 0.1], np.full(6, 1 / 6), p4]:
    print(np.round(p, 3), H_bits(p))

- 公平硬幣 1 bit；偏成 0.9 的硬幣只有 0.469 bit（大多是正面，不太需要問）；一定會發生的事是 0。
- **$K$ 種結果的熵最小是 0（一定是某一個），最大是 $\log K$**：每個一樣可能，叫**均勻分佈 uniform distribution**。熵量的是**不確定性**：分佈越平越大、越集中越小（證明用工具三的不等式，拿 $q$ 當均勻分佈）。
- 硬幣的熵畫成 $p$ 的函數（白努利分佈的熵），是一座中間最高的山：$p = 0.5$ 時 1 bit，兩端是 0（本集重點那張圖的左邊）。

（網站上有互動元件：拖動 4 個 token 的機率，看每個 token 的驚訝程度（要問幾題）和熵，可以切換 bit／nat；機率 0 的 token 顯示「∞（不會發生）」，不算進熵。）

## 3. 工具三：交叉熵

### 照錯的分佈猜

回到工具二的猜謎，真正的機率是 $p$＝（冷 $\frac{1}{2}$、熱 $\frac{1}{4}$、晴 $\frac{1}{8}$、暖 $\frac{1}{8}$），可是你**以為**機率是 $q$，就會照 $q$ 安排問法：每個 token 問 $-\log_2 q(x)$ 題。真正平均要問幾題？每一行是一個 $q$、照 $q$ 每個 token 要問幾題，以及用真正的 $p$ 加權的平均：

In [ ]:
q_uniform = np.full(4, 0.25)                   # 以為四個一樣可能
q_wrong = np.array([0.125, 0.125, 0.25, 0.5])  # 以為「暖」最常見
for q in [p4, q_uniform, q_wrong]:
    print(q, -np.log2(q), p4 @ -np.log2(q))    # 照 q 每個 token 問幾題；用真正的 p4 平均

照對的分佈猜，平均 1.75 題（就是熵）；以為一樣可能，2 題；以為「暖」最常見，2.625 題。這個數叫**交叉熵 cross-entropy**：

$$
H(p, q) = \sum_x p(x)\,\bigl(-\log q(x)\bigr)
$$

白話：驚訝程度用 $q$ 算，平均用 $p$ 加權。

$H(p, q)$ 念「$p$ 和 $q$ 的交叉熵」，**順序有意義**：$p$ 是真正會發生的（資料），$q$ 是你相信的（模型），兩個分佈交叉著用；$q = p$ 時就是熵。

### 交叉熵不會比熵小

上面三個 $q$，照對的猜最省。這是一般的事實，叫**吉布斯不等式 Gibbs inequality**：

$$
H(p, q) \ge H(p)\text{，只有 } q = p \text{ 時相等}
$$

白話：照錯的分佈猜，平均一定要多問。

理由只要高中數學：$\ln x \le x - 1$，只在 $x = 1$ 相等（$y = x - 1$ 是 $\ln x$ 在 $x = 1$ 的切線，$\ln x$ 一路往下彎，整條都在切線下面）。用 $\ln$ 算兩者的差，只加 $p(x) > 0$ 的項：

$$
\begin{aligned}
H(p, q) - H(p) &= \sum_x p(x) \ln \frac{p(x)}{q(x)} = -\sum_x p(x) \ln \frac{q(x)}{p(x)} \\
&\ge -\sum_x p(x) \Bigl(\frac{q(x)}{p(x)} - 1\Bigr) = 1 - \sum_x q(x) \ge 0
\end{aligned}
$$

白話：把每個 $\ln \frac{q(x)}{p(x)}$ 換成不比它小的 $\frac{q(x)}{p(x)} - 1$，前面有負號，整個式子只會變小；變小之後是「1 減掉一些 $q(x)$」，還是 $\ge 0$，原來的差當然也 $\ge 0$。

要相等，每一項的 $\frac{q(x)}{p(x)}$ 都要是 1，也就是 $q = p$。所以**交叉熵越小，$q$ 越像 $p$**，這就是它能當損失的原因（第 8 節的檢查會用 100 個隨機的分佈檢查這件事）。拿 $q$ 當 $K$ 種結果的均勻分佈：每個 $-\log q(x)$ 都是 $\log K$，$H(p, q) = \log K$，所以 $H(p) \le \log K$，這就是工具二「均勻分佈的熵最大」的理由。

### 正確答案只有一個：one-hot

訓練資料裡的每個位置，下一個 token 是確定的。把正確答案寫成分佈：正確的那一格是 1、其他都是 0，叫 **one-hot**。例如 EP00 的六格裡正確答案是「冷」：$y = (1, 0, 0, 0, 0, 0)$。交叉熵只剩一項：

$$
H(y, q) = \sum_x y(x)\,\bigl(-\log q(x)\bigr) = -\log q(\text{正確 token})
$$

白話：正確答案只有一個時，交叉熵就是模型給正確答案的機率取 $-\log$。

印出這個交叉熵，左邊用 nat、右邊用 bit：

In [ ]:
q_ep00 = np.array([0.293, 0.229, 0.065, 0.045, 0.033, 0.335])   # 模型 q：EP00 的冷 熱 晴 暖 乾 其他（EP06）
y = np.array([1, 0, 0, 0, 0, 0])               # 正確答案是「冷」：one-hot
print(y @ -np.log(q_ep00), y @ -np.log2(q_ep00))   # nat、bit

交叉熵 1.228 nat，就是「冷」的資訊量。EP02 說「LLM 的損失就是 $-\log p$」，指的就是這個。

## 4. 工具四：KL 散度

### 交叉熵 − 熵

工具三那個差有自己的名字，叫 **KL 散度 KL divergence**：

$$
D_{\mathrm{KL}}(p \,\|\, q) = H(p, q) - H(p) = \sum_x p(x) \log \frac{p(x)}{q(x)}
$$

白話：**資料是 $p$，卻照 $q$ 來猜，平均多付出的題數**（用 $\log_2$ 時單位是 bit）。

念「$p$ 對 $q$ 的 KL 散度」，中間的兩條直線只是隔開資料 $p$ 和模型 $q$（不是絕對值）；KL 是提出它的兩個人 Kullback 和 Leibler，散念「散步」的散。由吉布斯不等式，它 $\ge 0$，只有 $q = p$ 時是 0。印出「以為一樣可能」和「以為『暖』最常見」的 KL（bit）：

In [ ]:
def KL_bits(p, q):
    return np.sum(p * np.log2(p / q))           # 先假設沒有 0

print(KL_bits(p4, q_uniform), KL_bits(p4, q_wrong))

0.25 和 0.875，就是 $2 - 1.75$ 和 $2.625 - 1.75$：平均多問 0.25 題和 0.875 題。畫成堆疊長條（下段都是熵 1.75，上段是 KL，整根是交叉熵；圖上的 warm 是「暖」）：

In [ ]:
guesses = [("guess with p\n(the truth)", p4),             # 照真正的 p 猜
           ("think all 4\nequally likely", q_uniform),     # 以為四個一樣可能
           ("think 'warm'\nis most common", q_wrong)]      # 以為「暖」最常見
h4 = p4 @ -np.log2(p4)                                     # 熵 1.75 bit
ces4 = np.array([p4 @ -np.log2(g) for _, g in guesses])    # 三種猜法的交叉熵
kls4 = ces4 - h4                                           # KL：多問的題數
xs = np.arange(len(guesses))
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(xs, [h4] * len(guesses), width=0.55, color=BLUE, edgecolor="white", linewidth=2,
       label=f"entropy H(p) = {h4:g} bit")
ax.bar(xs, kls4, bottom=h4, width=0.55, color=RED, edgecolor="white", linewidth=2,
       label="KL(p || q): extra questions")
for xi, ce4, kl4 in zip(xs, ces4, kls4):
    ax.text(xi, ce4 + 0.05, f"{ce4:g}" + (f"  (+{kl4:g})" if kl4 > 0 else ""), ha="center", va="bottom",
            fontsize=10, color=INK)
ax.set_xticks(xs, [label for label, _ in guesses])
ax.set(ylim=(0, 3.2), ylabel="average questions (bit)", title="cross-entropy H(p, q) = entropy + KL")
ax.legend(loc="upper left", frameon=False)
tidy(ax, axis="y")
plt.show()

### 不對稱：不是距離

資料是公平硬幣、模型以為偏成 0.9，和反過來。左邊是 $D_{\mathrm{KL}}(\text{公平} \,\|\, \text{偏})$，右邊順序反過來（bit）：

In [ ]:
fair, biased = np.array([0.5, 0.5]), np.array([0.9, 0.1])
print(KL_bits(fair, biased), KL_bits(biased, fair))   # 順序反過來

$D_{\mathrm{KL}}(p \,\|\, q) \ne D_{\mathrm{KL}}(q \,\|\, p)$：「硬幣其實公平，你以為偏成 0.9」多付 0.737 bit；「硬幣其實偏，你以為公平」多付 0.531 bit。距離應該兩個方向一樣，所以叫「散度」不叫距離。**順序一定要對**：$p$ 是資料、$q$ 是模型（弄壞它 3）。

### 資料會出現、模型卻給 0：無限大

EP06 弄壞它 4：丟 3 次都是正面，最大概似估計說 $\hat{q}$＝（正面 1、反面 0）。拿它去猜一枚真正公平的硬幣 $p$：反面那一項是 $0.5 \log_2 \frac{0.5}{0}$，$D_{\mathrm{KL}}(p \,\|\, \hat{q}) = \infty$，一次反面就「無限驚訝」。

所以語言模型**不能給資料裡會出現的 token 剛好 0 的機率**。Softmax 的輸出永遠大於 0（$e^z > 0$），電腦上卻會下溢（弄壞它 1）；用計數做的模型會給 0，EP17 用平滑處理。

### 資料固定時：壓低交叉熵＝壓低 KL

把定義移項：

$$
H(p, q) = H(p) + D_{\mathrm{KL}}(p \,\|\, q)
$$

白話：交叉熵＝資料本身的熵＋模型猜錯多付的部分。

訓練時資料 $p$ 是固定的，$H(p)$ 是常數，模型能改的只有 KL。所以**把交叉熵壓低，就是把 KL 壓低，讓模型 $q$ 靠近資料 $p$**，最低降到 $H(p)$。熵那一份是資料本身的不確定性（「今天天氣很」後面本來就可能接很多字），模型壓不掉。

## 5. 工具五：困惑度

### 把熵換回「幾個選項」

$K$ 個一樣可能時，熵是 $\log_2 K$ bit，反過來 $K = 2^{\text{熵}}$。對任何分佈都這樣算，叫**困惑度 perplexity**：

$$
\text{困惑度} = 2^{H \text{（bit）}} = e^{H \text{（nat）}}
$$

白話：**這個分佈的不確定性，相當於在幾個一樣可能的選項之間猶豫**。

第一行是偏成 0.9 的硬幣、工具二四個 token 的困惑度；第二行依序是「151,936 個一樣可能」的困惑度、EP06 示範句的損失（nat）和它的困惑度：

In [ ]:
print(2 ** H_bits([0.9, 0.1]), 2 ** H_bits(p4))       # 偏的硬幣、工具二的四個 token
loss_demo = -np.mean(np.log([0.01, 0.2, 0.9, 0.3, 0.293]))   # EP06 的示範句：取負號、除以 5
print(np.exp(np.log(V)), loss_demo, np.exp(loss_demo))

公平硬幣是 $2^1 = 2$、公平骰子是 6；偏成 0.9 的硬幣只像在 1.38 個選項之間猶豫，工具二的四個 token 像 3.36 個。最小是 1（確定）。兩種寫法一樣：$h$ nat 是 $\frac{h}{\ln 2}$ bit，而 $2 = e^{\ln 2}$，所以 $2^{h / \ln 2} = e^{h}$。

**模型的困惑度**用交叉熵算：平均交叉熵（就是損失，本集重點）是 $\text{loss}$ nat，困惑度就是 $e^{\text{loss}}$，越低越好。它量的**不是**模型多猶豫：一個 3 個 token 的模型每次都給 (0.001, 0.998, 0.001)，很有把握（熵只有 0.023 bit），正確答案卻每次都是只拿到 0.001 的那一個（弄壞它 2），損失 $-\ln 0.001 \approx 6.91$ nat，困惑度是 1000。

### 兩個要記住的數

上面那格的第二行：

- **什麼都沒學的模型**：151,936 個 token 一樣可能，每個位置的交叉熵都是 $\ln 151{,}936 \approx 11.93$ nat，困惑度 151,936（`151935.99999999997` 是捨入誤差）。
- **EP06 的示範句**「今天／天／氣／很／冷」：每一步正確 token 的機率是 0.01、0.2、0.9、0.3、0.293，log 機率加起來 −8.752，取負號、除以 5 個 token，損失 1.750 nat，困惑度 $e^{1.750} \approx 5.76$（只有 0.293 是 EP00 實測，前四個是示範數字，所以 5.76 也只是示範）。5 個條件機率連乘約是 $\bigl(\frac{1}{5.76}\bigr)^5$，就像每一步都從 5.76 個一樣可能的 token 裡猜中一個：正確 token 平均（連乘再開 5 次方根，叫**幾何平均 geometric mean**）拿到約 $0.174 \approx \frac{1}{5.76}$。

困惑度只能在**同一份資料、同一個 tokenizer** 之間比：token 切法不同，「每個 token」的意思就不同（EP19）。

（網站上有互動元件：調一串正確 token 的機率（預設是示範句），或切換成「K 個一樣可能」，看平均損失、困惑度，和「正確 token 平均拿到約 1/困惑度」。）

## 6. 本集重點：為什麼 LLM 的損失是交叉熵

### 每個位置：−log q(正確 token)

訓練資料裡的每個位置，模型看了前文，替每個 token 打一個分數 $z$（EP00 的 logits），機率 $q = \text{Softmax}(z)$；正確答案是 one-hot。工具三說過，這個位置的交叉熵就是 $-\log q(\text{正確 token})$。用 EP02 的 log-softmax 算：

$$
-\ln q_{\text{正確}} = -\bigl(z_{\text{正確}} - \text{logsumexp}(z)\bigr) = \text{logsumexp}(z) - z_{\text{正確}}
$$

白話：全部分數的 logsumexp，減掉正確答案的分數。

logsumexp 用的是 $e$，所以 LLM 的損失單位是 nat。全部 $N$ 個位置平均：

$$
\text{loss} = \frac{1}{N} \sum_{t=1}^{N} \bigl(-\ln P_\theta(x_t \mid x_{<t})\bigr) = -\frac{\text{log 概似}}{N}
$$

白話：每個位置「正確的下一個 token 的機率」取 $-\ln$，加起來再除以位置數。

$P_\theta(x_t \mid x_{<t})$ 是 EP06 的寫法：參數 $\theta$ 的模型看了前面的 token $x_{<t}$，給第 $t$ 個 token $x_t$ 的機率，也就是那個位置的 $q_{\text{正確}}$。**這就是 EP06 說的「取負號、再除以 token 數」**：log 概似取負號叫**負 log 概似 NLL negative log-likelihood**，「越大越好」變成「越小越好」；除以 $N$，資料多少都能比。

用 EP06 的玩具模型（3 個 token、不看前文、資料是那 10 個 token），起點三個分數都是 0。第一行是每個位置的交叉熵；第二行依序是總和、平均，以及用 logsumexp 減掉正確分數算的第一個位置：

In [ ]:
data = np.array([0, 1, 0, 2, 0, 2, 2, 0, 1, 0])   # EP06：冷 熱 冷 好 冷 好 好 冷 熱 冷（0 冷、1 熱、2 好）
z = np.zeros(3)                                    # 起點：三個分數一樣
per_token = -log_softmax(z)[data]                  # 每個位置的交叉熵 −log q(正確 token)
print(np.round(per_token, 4))
print(per_token.sum(), per_token.mean(), logsumexp(z) - z[0])

總和 10.986 就是 EP06 的 log 概似 −10.986 去掉負號，平均 1.0986 就是損失；最後一個數用 logsumexp 減掉正確分數算第一個位置，一樣。

### 平均起來：資料和模型的交叉熵

10 個位置裡，「冷」5 次、「熱」2 次、「好」3 次。把同一個 token 的位置放在一起加：

$$
\text{loss} = \frac{1}{N} \sum_{t=1}^{N} \bigl(-\log q(x_t)\bigr) = \sum_j \frac{c_j}{N} \bigl(-\log q_j\bigr) = H(\hat{p}, q)
$$

白話：10 個位置的平均，等於「每個 token 的比例 × 它的 $-\log q$」加起來。

$c_j$ 是第 $j$ 個 token 出現的次數，$\hat{p}_j = \frac{c_j}{N}$ 是資料裡的比例（EP06 的經驗分佈）。**平均損失正好是資料 $\hat{p}$ 和模型 $q$ 的交叉熵**，這就是它叫交叉熵的原因。

第一行依序是資料的比例、交叉熵、熵、KL（都是 nat）；第二行是交叉熵和熵的困惑度：

In [ ]:
p_hat = np.bincount(data, minlength=3) / data.size   # 資料的比例
q = softmax(z)
ce = p_hat @ -np.log(q)                              # 交叉熵 H(p̂, q)（nat）
H = p_hat @ -np.log(p_hat)                           # 熵 H(p̂)
print(p_hat, ce, H, ce - H)
print(np.exp(ce), np.exp(H))                         # 困惑度

EP06 把機率訓練到資料的比例 $q = \hat{p}$：

| | 起點 $q = (\frac{1}{3}, \frac{1}{3}, \frac{1}{3})$ | 訓練到 $q = \hat{p} = (0.5, 0.2, 0.3)$ |
|---|---|---|
| 損失＝交叉熵 $H(\hat{p}, q)$ | 1.0986（＝ EP06 的 −10.986 取負號 ÷ 10） | 1.0297（＝ EP06 的 −10.2965 取負號 ÷ 10） |
| 熵 $H(\hat{p})$ | 1.0297 | 1.0297 |
| KL 散度 $D_{\mathrm{KL}}(\hat{p} \,\Vert\, q)$ | 0.0690 | 0 |
| 困惑度 | 3 | 2.80 |

四件事是同一件事：**壓低損失＝壓低交叉熵＝最大概似估計（EP06）＝壓低 KL，讓模型靠近資料**。損失降不到 0，最低是資料的熵 1.0297：資料本身就有不確定性。把 EP06 訓練時每一步的 $q$ 都拿來算，畫成堆疊長條，熵那一層不動，KL 那一層縮到看不見。

這張圖（左邊另外畫了工具二說的硬幣的熵）由第 8 節動手做的 `bernoulli_entropy_bits`、`draw_bernoulli`、`draw_ce_stack` 來畫，先自己寫寫看（參考解答在最後）。

（網站上有互動元件：資料固定是 $\hat{p} = (0.5, 0.2, 0.3)$，拖動模型的 $q$，看「熵＋KL＝交叉熵」的堆疊長條，熵那一層不動；也可以按「用負梯度走一步」，看 KL 縮到 0。）

真正的語言模型每個位置的前文不一樣，$q$ 也不一樣：每一種前文各自是一個交叉熵，再用前文出現的比例加權平均。KL 的順序也不能反過來：$D_{\mathrm{KL}}(q \,\|\, \hat{p})$ 裡有 $\log \frac{q}{\hat{p}}$，前文後面沒出現過的 token 比例 $\hat{p}$ 是 0，Softmax 的 $q$ 卻大於 0，結果是無限大。

## 7. Softmax 的導數：梯度為什麼是 q − y

### Softmax 的 Jacobian

要用梯度往下走，得知道損失對分數的導數，先推 Softmax 的。它的每個輸出都和**所有**分數有關（分母是全部加起來），Jacobian 是 $K \times K$ 的方陣，第 $i$ 列第 $j$ 欄是 $\frac{\partial q_i}{\partial z_j}$。取 log 就好推：

1. $\ln q_i = z_i - \text{logsumexp}(z)$（EP02 的 log-softmax）。
2. $\text{logsumexp}(z) = \ln \sum_k e^{z_k}$ 對 $z_j$ 微分：外層 $\ln u$ 的導數是 $\frac{1}{u}$，內層 $\sum_k e^{z_k}$ 只有 $e^{z_j}$ 這一項和 $z_j$ 有關，導數是 $e^{z_j}$；鏈鎖律（EP05）相乘得 $\frac{e^{z_j}}{\sum_k e^{z_k}} = q_j$。**logsumexp 的導數就是 Softmax。**
3. $z_i$ 對 $z_j$ 的導數是「$i = j$ 時 1，否則 0」，寫成 $\delta_{ij}$（念「delta i j」，叫 **Kronecker delta**，就是 `np.eye` 的第 $i$ 列第 $j$ 欄）。所以 $\frac{\partial \ln q_i}{\partial z_j} = \delta_{ij} - q_j$。
4. 左邊再用一次鏈鎖律：$\frac{\partial \ln q_i}{\partial z_j} = \frac{1}{q_i} \frac{\partial q_i}{\partial z_j}$。兩邊乘上 $q_i$：

$$
\frac{\partial q_i}{\partial z_j} = q_i\,(\delta_{ij} - q_j), \qquad J = \operatorname{diag}(q) - q\,q^\top
$$

白話：對角線是 $q_i(1 - q_i)$，其他格是 $-q_i q_j$。

$\operatorname{diag}(q)$ 是對角線放 $q$ 的對角矩陣（EP05），$q\,q^\top$ 是第 $i$ 列第 $j$ 欄為 $q_i q_j$ 的方陣（`np.outer(q, q)`，不是內積）。用 EP02 看過的分數 [1, 2, 3]，依序印出機率、Jacobian、和 EP05 數值 Jacobian 的最大差距，以及 Jacobian 每一列、每一欄的和：

In [ ]:
z3 = np.array([1.0, 2.0, 3.0])
q3 = softmax(z3)
J = np.diag(q3) - np.outer(q3, q3)                 # 第 i 列第 j 欄：q_i (δ_ij − q_j)
print(np.round(q3, 4))
print(np.round(J, 4))
print(np.abs(J - numerical_jacobian(softmax, z3)).max())   # 和 EP05 的數值 Jacobian 比
print(np.round(J.sum(axis=1), 12), np.round(J.sum(axis=0), 12))

- 和中央差分只差 $6 \times 10^{-12}$（中央差分本身的誤差）。
- 對角線是正的：自己的分數變大，自己的機率變大。其他格是負的：別人的機率被擠小。**每個輸出都和所有輸入有關**，就是這些不是 0 的格子。
- 每一列加起來是 0：第 $i$ 列的和是所有分數一起加一點點時 $q_i$ 的變化，而所有分數一起加同一個數，Softmax 不變（EP01）。每一欄加起來也是 0：機率的總和永遠是 1。

畫成熱圖（藍色是正的、紅色是負的，每格寫上數字）：

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

cmap = LinearSegmentedColormap.from_list("div", [RED, "#EDEDED", BLUE])   # 負的紅、0 淡灰、正的藍
fig, ax = plt.subplots(figsize=(5.6, 4.6))
im = ax.imshow(J, cmap=cmap, vmin=-0.25, vmax=0.25)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{J[i, j]:+.4f}", ha="center", va="center", fontsize=12, color=INK)
ax.set_xticks(range(3), [f"z{j + 1} = {z3[j]:g}" for j in range(3)])
ax.set_yticks(range(3), [f"q{i + 1} = {q3[i]:.3f}" for i in range(3)])
ax.set(xlabel="score moved (column j)", ylabel="probability that responds (row i)",
       title="Softmax Jacobian dq_i/dz_j = q_i (delta_ij - q_j)")
ax.tick_params(length=0)
for side in ax.spines.values():
    side.set_visible(False)
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.outline.set_visible(False)
ax.text(1, 3.0, "every row and every column sums to 0", ha="center", va="top", fontsize=10, color=MUTED)
plt.show()

### 損失對分數的梯度：q − y

一個位置的損失 $\text{loss} = -\ln q_{\text{正確}}$ 對 $q$ 的梯度是一列，只有正確那一格不是 0：$-\frac{1}{q_{\text{正確}}}$。用 EP05 的「一列 × Jacobian」，乘出來就是 $-\frac{1}{q_{\text{正確}}}$ 乘上 $J$ 的「正確」那一列：

$$
\frac{\partial\,\text{loss}}{\partial z_j} = -\frac{1}{q_{\text{正確}}} \cdot q_{\text{正確}}\,(\delta_{\text{正確},j} - q_j) = q_j - y_j
$$

白話：**損失對分數的梯度是 $q - y$：預測的機率減掉正確答案的 one-hot**。$\delta_{\text{正確},j}$ 在 $j$ 是正確答案時是 1、否則是 0，正好就是 one-hot 的 $y_j$。

直接微分 $\text{logsumexp}(z) - z_{\text{正確}}$ 也一樣：前項的導數是 $q$，後項是 $y$。第一行是「一列 × Jacobian」；第二行左邊是公式 $q - y$，右邊是中央差分：

In [ ]:
y3 = np.eye(3)[0]                                  # 正確答案是第 0 個：one-hot
print(-1 / q3[0] * J[0])                           # 一列 × Jacobian：−1/q_正確 乘上 J 的第 0 列
print(q3 - y3, numerical_gradient(lambda v: -log_softmax(v)[0], z3))   # 公式、中央差分

正確答案（機率 0.09）的梯度是 $0.09 - 1 = -0.91$：往負梯度走，它的分數被往上推；另外兩個被往下壓，壓多少看它們的機率。

### 兌現 EP06：實際次數 − 預期次數

玩具模型 10 個位置共用同一組分數 $z$，平均損失的梯度是 10 個 $\frac{1}{10}(q - y_t)$ 加起來：

$$
\frac{1}{10} \sum_{t=1}^{10} (q - y_t) = q - \hat{p}
$$

白話：10 個 one-hot 的平均，就是資料裡每個 token 的比例 $\hat{p}$。

EP06 是讓 log 概似**變大**；log 概似 $= -10 \times \text{loss}$，所以 EP06 的梯度是 $-10 \times (q - \hat{p}) = 10\hat{p} - 10q$，就是**實際次數 − 預期次數**，只差正負號和倍數。第一行是起點的梯度 $q - \hat{p}$ 和它乘上 $-10$；中間七行依序是步數、機率、損失、KL、困惑度；最後一行是冷、熱、好各自的 $-\log q$：

In [ ]:
grad = softmax(np.zeros(3)) - p_hat                # 平均損失對共用分數的梯度：q − p̂
print(grad, -10 * grad)
z = np.zeros(3)
for step in range(51):
    q = softmax(z)
    ce = p_hat @ -np.log(q)                        # 損失＝交叉熵
    if step in [0, 1, 2, 5, 10, 20, 50]:
        print(step, np.round(q, 3), round(ce, 4), round(ce - H, 4), round(np.exp(ce), 3))
    z = z - 1.0 * (q - p_hat)                      # 往負梯度走一步
print(np.round(-np.log(softmax(z)), 4))            # 最後：冷、熱、好各自的 −log q

- 第一行乘 $-10$ 就是 EP06 的 $(1.67, -1.33, -0.33)$。
- 每一步減掉 1 倍的梯度，等於 EP06 加上 0.1 倍的 log 概似梯度（$0.1 \times 10 = 1$），所以機率和 EP06 每一步都一樣（一步該走多遠是 EP08 的事）。
- 損失降到熵 1.0297 就停了：梯度 $q - \hat{p}$ 在 $q = \hat{p}$ 時是 0，**這就是 EP06 的機率停在比例的原因**。
- 最後一行：「冷」的 $-\log q$ 從 1.0986 降到 0.693，「熱」反而**升到** 1.609（機率從 $\frac{1}{3}$ 降到 0.2），「好」也升到 1.204：10 個位置有 5 個變差，照比例分才讓平均最小。

### 為什麼不用別的損失

- **它就是最大概似估計**：「讓訓練資料的機率最大」換個寫法。
- **梯度 $q - y$ 在最需要改的時候推得最用力**：很有把握卻答錯（正確答案的機率接近 0），那一格的梯度接近 $-1$；很有把握地答對（$q \approx y$）就接近 0。log 的 $-\frac{1}{q_{\text{正確}}}$ 剛好抵掉 Jacobian 裡的 $q_{\text{正確}}$；換成**均方誤差 MSE mean squared error**（機率和 one-hot 的差距平方、再取平均，EP04 當示範用過），這種時候梯度會被 Jacobian 壓到幾乎是 0（弄壞它 2）。
- **算起來穩定**：$\text{logsumexp}(z) - z_{\text{正確}}$ 不會下溢（EP02、弄壞它 1）。

（網站上有互動元件：調 3 個分數、點選正確答案，看 Softmax 的機率、交叉熵損失，和負梯度 $y - q$ 的箭頭（每個分數該往哪邊推、推多少）；可以切換成均方誤差，看很有把握卻答錯時箭頭幾乎消失。）

### 和 LLM 的連結

- PyTorch 的 `F.cross_entropy(logits, targets)`：`logits` 形狀 `(N, V)`，直接吃分數，內部做 log-softmax、取出正確 token、取平均，回傳 nat。LLM 的分數是 `(B, T, V)`（$B$ 句、每句 $T$ 個位置），攤平成 `(B·T, V)` 再丟進去，每個位置的正確答案是**下一個** token。
- 像 Qwen2.5-0.5B 這樣的模型，預訓練的損失就是平均每個 token 的交叉熵（nat）。剛初始化時如果每個 token 的分數都差不多，損失會在 $\ln 151{,}936 \approx 11.93$ 附近，可以拿來檢查訓練的第一步。
- 反向傳播的起點就是 $q - y$，再往回一路乘 Jacobian（EP05）。EP12 用 Softmax＋交叉熵辨認手寫數字，EP17 的 bigram 也算這個損失。

## 8. 動手做：從零寫出 `info.py` 的 6 個函式

在下一格寫完這 6 個函式（網站上是寫在 `llm-labs/ep07/info.py`），再執行下一格的檢查。

要求與提示：

1. **只用 NumPy、基本的 Python 和前幾集自己寫的函式**：`softmax`、`log_softmax` 已經定義在最上面，直接用（網站上是 `from ep01.softmax import softmax`、`from ep02.mathlang import log_softmax`）。
2. **`entropy`、`cross_entropy`、`kl_divergence`**：先 `np.asarray(..., dtype=float)`（檢查會傳 list），用 `m = p > 0` 只取 $p > 0$ 的項，`np.log` 就不會遇到 0；有 $p > 0$ 而 $q = 0$ 的項（`np.any(q[m] == 0)`）就直接 `return np.inf`。`base` 用換底：nat 除以 `np.log(base)`。`kl_divergence` 也可以寫成 `cross_entropy − entropy`。`np.asarray` 遇到小數陣列不會複製，不要在上面改值（例如 `p[p == 0] = 1`），會改到呼叫端的陣列（同 EP06）。
3. **`cross_entropy_loss`**：`a = log_softmax(logits, axis=-1)` 是 `(N, K)`，`a[np.arange(N), targets]` 一次取出每一列正確 token 那一格。不要寫成 `np.log(softmax(...))`（弄壞它 1）。一定要檢查 `targets`：索引 `-1` 是最後一個（EP01），寫錯只會默默取錯格。整數檢查用 `np.issubdtype(targets.dtype, np.integer)`；`targets` 先 `np.asarray`，別加 `dtype=int`（會把 `[0.0, 1.0]` 默默轉成整數）。
4. **`softmax_ce_grad`**：one-hot 用 `np.eye(K)[targets]`（恆等矩陣的每一列就是一個 one-hot），記得除以 $N$。

In [ ]:
# softmax（EP01）、log_softmax（EP02）已經定義在最上面，直接用。


def entropy(p, base=None):
    """熵 H(p) = Σ p (−log p)，p 是一維的機率分佈。base=None 用 ln（nat），base=2 用 log₂（bit）。
    0 × log 0 當成 0：不能得到 nan，也不能跳出 RuntimeWarning。回傳 float。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 entropy")


def cross_entropy(p, q, base=None):
    """交叉熵 H(p, q) = Σ p (−log q)（p 是資料、q 是模型，同長度）。p = 0 的項當成 0；
    有一項 p > 0 而 q = 0，回傳 np.inf，不能跳出 RuntimeWarning。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 cross_entropy")


def kl_divergence(p, q, base=None):
    """KL 散度 D(p ‖ q) = Σ p log(p / q)。0 的處理和 cross_entropy 一樣。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 kl_divergence")


def cross_entropy_loss(logits, targets):
    """logits：(N, K) 的分數；targets：長度 N 的整數（每個位置正確 token 的 ID）。回傳平均交叉熵
    （nat，float），和 PyTorch 的 F.cross_entropy 一樣。logits 不是二維、targets 長度不對、
    不是整數、或不在 0 到 K − 1，raise ValueError。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 cross_entropy_loss")


def softmax_ce_grad(logits, targets):
    """cross_entropy_loss 對 logits 的梯度 (softmax(logits) − one-hot(targets)) / N，形狀 (N, K)。檢查同上。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 softmax_ce_grad")


def perplexity(loss):
    """困惑度 exp(loss)，loss 是平均交叉熵（nat）。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 perplexity")

下一格的 7 個檢查和網站上的 `tests/test_ep07_info.py` 一樣，依序檢查：

1. 熵：公平硬幣 1 bit、沒給 `base` 是 nat、100 個隨機分佈的熵都比均勻分佈小；$0 \times \log 0$ 當成 0 而且沒有警告，也不能改到呼叫端的陣列。
2. 交叉熵：100 個隨機分佈都比熵大（吉布斯不等式）、$q = p$ 時相等、one-hot 時是 $-\log q(\text{正確})$、工具三用 bit 的 2.625；$p$ 有機率而 $q$ 給 0 時是 $\infty$，$p = 0$ 的項不算。
3. KL 散度：公平硬幣和偏硬幣兩個順序的值、100 個隨機分佈都大於 0 而且等於交叉熵 − 熵、$q = p$ 時是 0、$q$ 給 0 時是 $\infty$。
4. `cross_entropy_loss`：等於平均的 $-\log q(\text{正確 token})$，也等於每個位置 one-hot 交叉熵的平均；分數差 1000 也不會下溢；`targets` 有 −1、超出範圍、不是整數、長度不對，以及 `logits` 不是二維，都要報錯（梯度也要檢查）。
5. `softmax_ce_grad`：和中央差分的梯度一樣（這就是 EP04 說的梯度檢查，EP11 會拿它檢查自動微分），每一列加起來是 0。
6. EP06 的玩具模型：起點的損失是 $\ln 3$、熵是 1.0297、損失 − 熵＝ KL；梯度乘 $-10$ 是實際次數 − 預期次數；$q = \hat{p}$ 時損失降到熵、梯度是 0；困惑度。
7. 和 PyTorch 比對（沒裝 PyTorch 就跳過）。

幾個寫法：

- `with raises(ValueError):` 是「底下這段程式應該要報 `ValueError`」，沒報錯，檢查就失敗。（網站上的測試用 pytest 的 `pytest.raises`，這裡寫了一個用法一樣的小工具，notebook 就不用裝 pytest。）
- `warnings.simplefilter("error")` 把警告變成錯誤；放在 `with warnings.catch_warnings():` 底下，只影響那一段。所以「不能跳出 `RuntimeWarning`」也檢查得到。
- 和 PyTorch 比對時兩邊只差捨入誤差，所以用很嚴的 `rel_tol=1e-12`。

沒通過的檢查會印出出錯的那一行。改了上一格之後，要重新執行上一格，再執行這一格。

In [ ]:
import contextlib
import traceback
import warnings


def check_info():
    """和網站上 tests/test_ep07_info.py 一樣的 7 個測試。"""

    class Skipped(Exception):
        pass

    @contextlib.contextmanager
    def raises(exc_type):
        """和 pytest.raises 一樣：底下這段程式應該要報 exc_type，沒報錯就算失敗。"""
        try:
            yield
        except exc_type:
            pass
        else:
            raise AssertionError(f"應該要報 {exc_type.__name__}，卻沒有報錯")

    DATA = np.array([0, 1, 0, 2, 0, 2, 2, 0, 1, 0])   # EP06 的 10 個 token：冷 5、熱 2、好 3
    P_HAT = np.array([0.5, 0.2, 0.3])
    UNIFORM3 = np.full(3, 1 / 3)

    def random_dist(rng, k):
        p = rng.random(k)
        return p / p.sum()

    def entropy_known_values():
        assert math.isclose(entropy([0.5, 0.5], base=2), 1.0)                 # 公平硬幣：1 bit
        assert math.isclose(entropy(np.full(6, 1 / 6)), math.log(6))          # 沒給 base：nat
        rng = np.random.default_rng(0)
        for _ in range(100):
            assert entropy(random_dist(rng, 5)) < math.log(5)                 # 均勻分佈最大
        with warnings.catch_warnings():
            warnings.simplefilter("error")                                    # 有警告就算失敗
            assert entropy([1.0, 0.0, 0.0]) == 0.0
            p0 = np.array([0.5, 0.0, 0.5])
            assert math.isclose(entropy(p0, base=2), 1.0)                     # 0 × log 0 當成 0
            cross_entropy(p0, p0), kl_divergence(p0, p0)
            assert np.array_equal(p0, [0.5, 0.0, 0.5])                        # 不能改到呼叫端的陣列

    def cross_entropy_at_least_entropy():
        rng = np.random.default_rng(1)
        for _ in range(100):
            p, q = random_dist(rng, 4), random_dist(rng, 4)
            assert cross_entropy(p, q) > entropy(p)                           # 吉布斯不等式
        assert math.isclose(cross_entropy(p, p), entropy(p))                  # q = p 時相等
        assert math.isclose(cross_entropy([0, 1, 0], [0.2, 0.7, 0.1]), -math.log(0.7))   # one-hot
        p4, q_wrong = [0.5, 0.25, 0.125, 0.125], [0.125, 0.125, 0.25, 0.5]
        assert math.isclose(cross_entropy(p4, q_wrong, base=2), 2.625)       # 工具三：用 bit
        with warnings.catch_warnings():
            warnings.simplefilter("error")
            assert cross_entropy([0.5, 0.5], [1.0, 0.0]) == np.inf           # p 有機率、q 給 0
            assert cross_entropy([1.0, 0.0], [1.0, 0.0]) == 0.0              # p = 0 的項不算

    def kl_divergence_properties():
        fair, biased = np.array([0.5, 0.5]), np.array([0.9, 0.1])
        assert math.isclose(kl_divergence(fair, biased), 0.5108256237659907)
        assert math.isclose(kl_divergence(biased, fair), 0.3680642071684971)
        rng = np.random.default_rng(2)
        for _ in range(100):
            p, q = random_dist(rng, 5), random_dist(rng, 5)
            assert kl_divergence(p, q) > 0
            assert math.isclose(kl_divergence(p, q), cross_entropy(p, q) - entropy(p))
        assert kl_divergence(p, p) == 0.0
        with warnings.catch_warnings():
            warnings.simplefilter("error")
            assert kl_divergence(fair, [1.0, 0.0]) == np.inf
            assert math.isclose(kl_divergence([1.0, 0.0], fair, base=2), 1.0)

    def cross_entropy_loss_is_mean_negative_log_prob():
        rng = np.random.default_rng(3)
        logits = rng.normal(size=(6, 4)) * 3
        targets = np.array([0, 3, 1, 1, 2, 0])
        loss = cross_entropy_loss(logits, targets)
        assert math.isclose(loss, -np.mean(log_softmax(logits)[np.arange(6), targets]))
        one_hot = [cross_entropy(np.eye(4)[t], softmax(z)) for z, t in zip(logits, targets)]
        assert math.isclose(loss, np.mean(one_hot))
        big = np.array([[0.0, 1000.0], [2.0, 1.0]])                           # 分數差 1000
        with warnings.catch_warnings():
            warnings.simplefilter("error")
            assert math.isclose(cross_entropy_loss(big, [0, 0]), (1000 + math.log(1 + math.exp(-1))) / 2)
        for bad in [[0, -1], [0, 2], [0.0, 1.0], [0]]:
            for f in [cross_entropy_loss, softmax_ce_grad]:                   # 梯度也要檢查
                with raises(ValueError):
                    f(big, bad)
        with raises(ValueError):
            cross_entropy_loss(big[0], [1])                                   # logits 不是二維

    def softmax_ce_grad_matches_numerical_gradient():
        rng = np.random.default_rng(4)
        logits = rng.normal(size=(5, 3))
        targets = np.array([2, 0, 0, 1, 2])
        grad = softmax_ce_grad(logits, targets)
        num = numerical_gradient(lambda z: cross_entropy_loss(z, targets), logits)
        assert grad.shape == (5, 3)
        assert np.allclose(grad, num, rtol=0, atol=1e-8)
        assert np.allclose(grad.sum(axis=1), 0.0)

    def toy_model_from_ep06():
        logits = np.zeros((10, 3))
        loss = cross_entropy_loss(logits, DATA)
        assert math.isclose(loss, math.log(3))                                # EP06 的 −10.986 取負號 ÷ 10
        assert math.isclose(loss, cross_entropy(P_HAT, UNIFORM3))
        assert math.isclose(entropy(P_HAT), 1.0296530140645737)               # EP06 的 −10.2965 取負號 ÷ 10
        assert math.isclose(loss - entropy(P_HAT), kl_divergence(P_HAT, UNIFORM3))
        shared = softmax_ce_grad(logits, DATA).sum(axis=0)                    # 10 個位置共用分數
        assert np.allclose(-10 * shared, [5 / 3, -4 / 3, -1 / 3])             # EP06：實際次數 − 預期次數
        best = np.tile(np.log(P_HAT), (10, 1))                                # q = p̂
        assert math.isclose(cross_entropy_loss(best, DATA), entropy(P_HAT))   # 降到熵，KL = 0
        assert np.allclose(softmax_ce_grad(best, DATA).sum(axis=0), 0.0)
        assert math.isclose(perplexity(loss), 3.0)
        assert math.isclose(perplexity(entropy(P_HAT)), 2 ** entropy(P_HAT, base=2))   # nat 用 e、bit 用 2

    def matches_pytorch():
        try:
            import torch
        except ImportError:
            raise Skipped("沒有安裝 PyTorch")
        F = torch.nn.functional
        rng = np.random.default_rng(5)
        logits = rng.normal(size=(6, 5)) * 3
        targets = np.array([4, 0, 2, 2, 1, 3])
        z = torch.tensor(logits, requires_grad=True)                         # float64
        loss = F.cross_entropy(z, torch.tensor(targets))
        loss.backward()
        assert math.isclose(loss.item(), cross_entropy_loss(logits, targets), rel_tol=1e-12)
        assert np.allclose(z.grad.numpy(), softmax_ce_grad(logits, targets), rtol=1e-12, atol=1e-15)
        p, q = softmax(logits[0]), softmax(logits[1])
        kl = F.kl_div(torch.log(torch.tensor(q)), torch.tensor(p), reduction="sum")   # 先 log q、再 p
        assert math.isclose(kl.item(), kl_divergence(p, q), rel_tol=1e-12)
        big = torch.tensor([[0.0, 1000.0]], dtype=torch.float64)
        assert F.cross_entropy(big, torch.tensor([0])).item() == cross_entropy_loss(big.numpy(), [0])

    tests = [entropy_known_values, cross_entropy_at_least_entropy, kl_divergence_properties,
             cross_entropy_loss_is_mean_negative_log_prob, softmax_ce_grad_matches_numerical_gradient,
             toy_model_from_ep06, matches_pytorch]
    passed = skipped = pending = 0
    for t in tests:
        try:
            t()
            print("✓", t.__name__)
            passed += 1
        except NotImplementedError as e:
            print("…", t.__name__, f"（{e}）")
            pending += 1
        except Skipped as e:
            print("-", t.__name__, f"跳過（{e}）")
            skipped += 1
        except Exception as e:
            msg = f"{type(e).__name__}: {e}" if str(e) else type(e).__name__
            frames = [fr for fr in traceback.extract_tb(e.__traceback__) if fr.name == t.__name__]
            line = frames[-1].line if frames else ""                    # 檢查裡出錯的那一行
            print("✗", t.__name__, msg, f"｜{line}" if line else "")
    summary = f"{passed}/{len(tests)} 通過"
    if skipped:
        summary += f"，{skipped} 個跳過"
    if pending:
        summary += f"，{pending} 個還沒寫好：先完成上一格，或執行最後的參考解答"
    print(summary)


check_info()

### 畫出熵與交叉熵：`bernoulli_entropy_bits`、`draw_bernoulli`、`draw_ce_stack`

網站上這是一個畫圖腳本 `llm-labs/ep07/plot_entropy.py`，執行後存成 `ep07/entropy.png`，也就是本集重點那張圖；在 notebook 裡只要在下一格寫好這 3 個函式，再下一格會用它們畫一列兩張圖（左邊 `draw_bernoulli`、右邊 `draw_ce_stack`），直接顯示。`DATA` 是 EP06 的 10 個 token，`STEPS` 是右圖要畫的步數。

要求與提示：

1. **熵、Softmax 和訓練迴圈都在這一格自己寫**，不呼叫上面的 `entropy`、`softmax`：網站上的腳本是直接執行的，Python 只在 `ep07/` 裡找模組，找不到 `ep01`、`ep07`（同 EP06）。自己寫的小函式別取名叫 `softmax` 或 `entropy`，會蓋掉上面的函式（例如叫 `softmax_1d`）。
2. **左圖**：`bernoulli_entropy_bits` 和 `entropy` 一樣，只算機率大於 0 的項（$p$ 和 $1 - p$ 各算一次）。用 `ax.plot([p], [h], "o")` 標出點，`ax.annotate(文字, (p, h))` 在旁邊寫出數字（1 bit、0.469 bit）。
3. **右圖**：堆疊長條先 `ax.bar(x, 熵)`，再 `ax.bar(x, KL, bottom=熵)` 疊上去，頂端用 `ax.text` 寫交叉熵（4 位小數，例如 `f"{ce:.4f}"`）；$y$ 軸從 0 開始，標籤寫上單位（左圖 bit、右圖 nat）。

In [ ]:
DATA = np.array([0, 1, 0, 2, 0, 2, 2, 0, 1, 0])     # EP06 的 10 個 token：冷 5、熱 2、好 3
STEPS = [0, 1, 2, 5, 10, 20]                        # 右圖要畫的步數


def bernoulli_entropy_bits(p):
    """白努利分佈的熵 H(p) = −p log₂ p − (1 − p) log₂(1 − p)（bit），p 可以是陣列；
    0 × log 0 當成 0，不能跳出警告。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 bernoulli_entropy_bits")


def draw_bernoulli(ax):
    """左圖：p 從 0 到 1（501 個點）畫 bernoulli_entropy_bits；標出 p = 0.5 和 p = 0.9 的點和數字。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 draw_bernoulli")


def draw_ce_stack(ax):
    """右圖：玩具模型的 z 從 0 開始，每一步換成 z − 1.0 × (q − p̂)；在 STEPS 那幾步畫堆疊長條（nat）：
    下段是熵 H(p̂)、上段是 KL，頂端寫交叉熵。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 draw_ce_stack")

下一格用你的 3 個函式畫圖，再檢查畫出來的東西。畫對的話就是本集重點那張圖：左圖是一座左右對稱的山，$p = 0.5$ 最高 1 bit、$p = 0.9$ 是 0.469 bit；右圖下段的熵都是 1.0297，上段的 KL 從 0.069 縮到看不見，頂端依序是 1.0986、1.0598、1.0430、1.0311、1.0297、1.0297。最後一個檢查換一份資料和步數（`DATA = [0, 0, 0, 1, 2, 2]`、`STEPS = [0, 3]`）再畫一次右圖（不顯示），確認數字是算出來的，不是寫死的；畫完會把 `DATA`、`STEPS` 改回來。

In [ ]:
import re
import warnings

from matplotlib.patches import Rectangle


def show_entropy_plot():
    """一列兩張圖：左邊 draw_bernoulli、右邊 draw_ce_stack，再檢查曲線、標出的點、堆疊長條、數字和座標軸。"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))
    try:
        draw_bernoulli(ax1)
        draw_ce_stack(ax2)
    except NotImplementedError as e:
        plt.close(fig)                  # 不顯示空白的圖
        print(f"{e}：先完成上一格，或執行最後的參考解答，再回來執行這格")
        return

    def H2(p):                                               # 檢查自己算一份：白努利分佈的熵（bit）
        p = np.asarray(p, dtype=float)
        with np.errstate(divide="ignore", invalid="ignore"):
            h = -p * np.log2(p) - (1 - p) * np.log2(1 - p)
        return np.where((p > 0) & (p < 1), h, 0.0)           # 兩端是 0 × log 0，當成 0

    def toy(data, steps):                                    # 檢查自己算一份：熵，和每個步數的（交叉熵, KL）
        p_hat = np.bincount(data, minlength=3) / data.size
        h = float(p_hat @ -np.log(p_hat))
        z, out = np.zeros(3), {}
        for step in range(max(steps) + 1):
            q = softmax(z)
            ce = float(p_hat @ -np.log(q))
            out[step] = (ce, ce - h)
            z = z - 1.0 * (q - p_hat)
        return h, [out[s] for s in steps]

    def bern_values():                                       # p = 0、0.5、0.9、1 的熵；有警告就算失敗
        with warnings.catch_warnings():
            warnings.simplefilter("error")
            return np.asarray(bernoulli_entropy_bits(np.array([0.0, 0.5, 0.9, 1.0])), dtype=float)

    def lines(ax):                                           # ax 上的每條線，排成 (2, N)
        return [np.array(l.get_data(), dtype=float) for l in ax.get_lines()]

    def curve(ax, f, lo, hi, n):                             # 有沒有一條線：x 從 lo 到 hi（n 個點），y 是 f(x)
        return any(d.shape[1] == n and np.isclose(d[0].min(), lo) and np.isclose(d[0].max(), hi)
                   and np.allclose(d[1], f(d[0]), rtol=1e-9, atol=1e-12) for d in lines(ax))

    def points(ax):                                          # 標出來的點：有 marker、點數很少的線，或 scatter
        pts = [(float(x), float(y)) for l in ax.get_lines()
               if l.get_marker() not in (None, "None", "", " ") and len(l.get_xdata()) <= 10
               for x, y in zip(l.get_xdata(), l.get_ydata())]
        for c in ax.collections:
            pts += [(float(x), float(y)) for x, y in c.get_offsets()]
        return pts

    def has_point(ax, x, y):
        return any(np.allclose(pt, (x, y), atol=1e-3) for pt in points(ax))

    def texts(ax):                                           # ax 上的文字，由左到右：（x, 文字）
        return sorted((float(t.xy[0]) if hasattr(t, "xy") else float(t.get_position()[0]), t.get_text())
                      for t in ax.texts)

    def says(ax, pattern):                                   # 有沒有一段文字符合 pattern
        return any(re.search(pattern, s) for _, s in texts(ax))

    def bars(ax):                                            # 每根長條：（中心 x, 底 y, 高度），由左到右
        return sorted((r.get_x() + r.get_width() / 2, r.get_y(), r.get_height())
                      for r in ax.patches if isinstance(r, Rectangle))

    def lower(ax, h):                                        # 從 0 開始、高度是熵 h 的長條
        return [b for b in bars(ax) if np.isclose(b[1], 0, atol=1e-12) and np.isclose(b[2], h, rtol=1e-9)]

    def upper(ax, h):                                        # 疊在熵 h 上面的長條（高度是 KL）
        return [b for b in bars(ax) if np.isclose(b[1], h, rtol=1e-9)]

    def stacked(ax, h, vals):                                # 上段和下段在同一個位置，高度依序是 KL
        lo, up = lower(ax, h), upper(ax, h)
        return (len(lo) == len(vals) and len(up) == len(vals)
                and np.allclose([b[0] for b in lo], [b[0] for b in up])
                and np.allclose([b[2] for b in up], [kl for _, kl in vals], rtol=1e-6, atol=1e-10))

    def tops(ax, vals):                                      # 由左到右依序寫出每一步的交叉熵（4 位小數）
        it = iter(s for _, s in texts(ax))
        return all(any(f"{ce:.4f}" in s for s in it) for ce, _ in vals)

    def rounded(bs):
        return [tuple(round(float(v), 4) for v in b) for b in bs]

    h_toy, vals_toy = toy(DATA, STEPS)
    other_data, other_steps = np.array([0, 0, 0, 1, 2, 2]), [0, 3]
    h_other, vals_other = toy(other_data, other_steps)
    other = {}

    def draw_other():                                        # 換一份資料和步數再畫一次右圖，不顯示；畫完改回來
        global DATA, STEPS
        if not other:
            saved = DATA, STEPS
            DATA, STEPS = other_data, other_steps
            fig2, b2 = plt.subplots()
            try:
                draw_ce_stack(b2)
            finally:
                DATA, STEPS = saved
                plt.close(fig2)
            other["ax"] = b2
        return other["ax"]

    checks = [
        ("bernoulli_entropy_bits：p = 0、0.5、0.9、1 是 0、1、0.469、0 bit（吃陣列、不跳出警告）",
         lambda: bern_values().shape == (4,) and np.allclose(bern_values(), [0, 1, H2(0.9), 0]),
         lambda: np.round(bern_values(), 4)),
        ("左圖：H(p) 的曲線，p 從 0 到 1（501 個點）",
         lambda: curve(ax1, H2, 0, 1, 501),
         lambda: "每條線的（點數, 最小 p, 最大 p, 有沒有 nan）"
                 + str([(d.shape[1], round(float(d[0].min()), 4), round(float(d[0].max()), 4), bool(np.isnan(d[1]).any()))
                        for d in lines(ax1)])),
        ("左圖：標出 p = 0.5 和 p = 0.9 兩個點",
         lambda: has_point(ax1, 0.5, 1.0) and has_point(ax1, 0.9, float(H2(0.9))),
         lambda: [tuple(round(v, 4) for v in pt) for pt in points(ax1)]),
        ("左圖：寫出數字 1 bit 和 0.469 bit",
         lambda: says(ax1, r"(?<![\d.])1(\.0+)?(?![\d.])") and says(ax1, r"0\.469"),
         lambda: [s for _, s in texts(ax1)]),
        ("左圖：y 軸標籤寫上單位 bit", lambda: "bit" in ax1.get_ylabel(), lambda: repr(ax1.get_ylabel())),
        (f"右圖：{len(STEPS)} 根下段，從 0 開始、高度都是熵 H(p̂) = {h_toy:.4f}",
         lambda: len(lower(ax2, h_toy)) == len(STEPS), lambda: rounded(bars(ax2))),
        ("右圖：上段疊在熵上面，高度依序是 KL " + "、".join(f"{kl:.4f}" for _, kl in vals_toy),
         lambda: stacked(ax2, h_toy, vals_toy), lambda: rounded(bars(ax2))),
        ("右圖：頂端由左到右寫出交叉熵 " + "、".join(f"{ce:.4f}" for ce, _ in vals_toy),
         lambda: tops(ax2, vals_toy), lambda: [s for _, s in texts(ax2)]),
        ("右圖：y 軸從 0 開始，標籤寫上單位 nat",
         lambda: np.isclose(ax2.get_ylim()[0], 0) and "nat" in ax2.get_ylabel(),
         lambda: (tuple(round(float(v), 3) for v in ax2.get_ylim()), ax2.get_ylabel())),
        ("換一份資料和步數（DATA = [0, 0, 0, 1, 2, 2]、STEPS = [0, 3]）：熵、KL、交叉熵跟著變",
         lambda: stacked(draw_other(), h_other, vals_other) and tops(draw_other(), vals_other),
         lambda: (rounded(bars(draw_other())), [s for _, s in texts(draw_other())])),
    ]
    plt.show()
    for name, ok, got in checks:
        try:
            passed = bool(ok())
        except Exception as e:                               # 例如畫圖時出錯，連比都沒辦法比
            passed, got = False, (lambda e=e: f"{type(e).__name__}: {e}")
        if passed:
            print("✓", name)
        else:
            try:
                print("✗", f"{name}：現在是 {got()}")
            except Exception as e:
                print("✗", f"{name}：{type(e).__name__}: {e}")


show_entropy_plot()

網站上的腳本把主程式放在 `if __name__ == "__main__":` 底下，用 `matplotlib.use("Agg")` 只存檔、不開視窗，並寫死 `figsize` 和 `dpi`，執行兩次的 `md5sum` 指紋一樣（同 EP04–EP06）；在 notebook 裡用 `plt.show()` 直接顯示就好。

### 和 PyTorch 對照

用你在上面寫的 `cross_entropy_loss`、`softmax_ce_grad`，和 PyTorch 對照（Colab 已經內建 PyTorch；沒裝的話這格會跳過）。會用到本集重點的 `data`。第一行是本集重點起點（10 個位置、分數都是 0）的損失，左邊 PyTorch、右邊你的函式；第二行是第一個位置（正確是「冷」）的梯度 $\frac{q - y}{10}$，左邊用 PyTorch 的 `.backward()` 算：

In [ ]:
try:
    import torch
    import torch.nn.functional as F

    logits = np.zeros((10, 3))                                   # 本集重點的起點：10 個位置、分數都是 0
    zt = torch.tensor(logits, requires_grad=True)                # float64
    loss = F.cross_entropy(zt, torch.tensor(data))               # 直接吃分數：內部做 log-softmax、取平均
    loss.backward()                                              # 倒著算回來（EP05）
    print(loss.item(), cross_entropy_loss(logits, data))
    print(zt.grad[0], softmax_ce_grad(logits, data)[0])          # 第一個位置（正確是「冷」）：(q − y) / 10
except ImportError:
    print("沒有安裝 PyTorch：跳過這格")
except NotImplementedError as e:
    print(f"{e}：先完成第 8 節的動手做，或執行最後的參考解答，再回來執行這格")

兩邊一樣（PyTorch 只印 4 位小數）。`F.cross_entropy` 吃的是**分數**：把 `softmax` 的輸出丟進去，它會再做一次 log-softmax，算出錯的數，也不會報錯。

## 9. 弄壞它

每一個都先**猜**結果，再執行，最後用自己的話解釋。完整的解釋在網站的「弄壞它」段落。

弄壞它 3 用到工具四的 `fair`、`biased` 和第 8 節的 `kl_divergence`，弄壞它 4 用到工具五的 `loss_demo`，弄壞它 5 用到第 8 節的 `entropy`。重新啟動過 kernel 的話，先重新執行第一格、定義前幾集函式的那一格，以及定義這些變數的格子。弄壞它 1、3 有一部分要用 PyTorch，沒裝的話會跳過那一部分，看下面的說明就好。

### 弄壞它 1：用 `np.log(softmax(z))` 算損失

EP02 看過，先做 Softmax 再取 log，log 機率會變成 `-inf`；這次看一整批的損失和梯度。第一個位置，模型很有把握是第 1 個，正確答案卻是第 0 個。第一行左邊是先 Softmax 再取 log 算的損失，右邊用 log-softmax；後面是 PyTorch 算出的兩種梯度（會跳出 `RuntimeWarning`：那只是警告，不會讓 notebook 停下來）：

In [ ]:
logits_b = np.array([[0.0, 1000.0],
                     [2.0, 1.0]])
targets_b = np.array([0, 0])
naive = -np.mean(np.log(softmax(logits_b))[np.arange(2), targets_b])   # 會跳出 RuntimeWarning
stable = -np.mean(log_softmax(logits_b)[np.arange(2), targets_b])
print(naive, stable)
try:
    import torch

    for name, f in [("log(softmax)", lambda t: torch.log(torch.softmax(t, dim=-1))),
                    ("log_softmax", lambda t: torch.log_softmax(t, dim=-1))]:
        zt = torch.tensor(logits_b, requires_grad=True)
        (-f(zt)[torch.arange(2), torch.tensor(targets_b)].mean()).backward()   # 讓 PyTorch 算梯度
        print(name, zt.grad)
except ImportError:
    print("沒有安裝 PyTorch：跳過比較梯度的部分（結果見下面的說明）")

第一行是 `inf 500.1566308437591`，還會跳出 `RuntimeWarning: divide by zero encountered in log`。$e^{-1000}$ 下溢成 0，正確答案的機率剛好是 0，log 是 `-inf`：**一個位置壞掉，整批的平均就是 `inf`**。梯度更糟：`log(softmax)` 那一批的第一列是 `nan`，因為 log 的導數 $\frac{1}{q} = \infty$ 乘上 Jacobian 裡的 $q = 0$，$\infty \times 0$ 是 `nan`，一進到參數，之後全部變 `nan`。log-softmax 的損失 $(1000 + 0.3133) / 2 = 500.16$ 很大但有限，梯度就是 $\frac{q - y}{N}$，第一列是 $(-0.5, 0.5)$。PyTorch 預設的 `float32`，分數差大約 104 就會這樣（EP02）。

### 弄壞它 2：用均方誤差當分類的損失

正確答案是第 0 個，模型只給它 0.001。拿均方誤差 MSE（機率和 one-hot 的差距平方、再取平均）和交叉熵 CE 比一開始的梯度，以及同樣步伐走幾步後正確答案的機率。第一行左邊是均方誤差的梯度、右邊是交叉熵的；後兩行是第 0、5、20、100、500 步正確答案的機率：

In [ ]:
z0 = np.array([0.0, np.log(998), 0.0])     # Softmax 剛好是 (0.001, 0.998, 0.001)
y0 = np.eye(3)[0]

def mse(z):
    return np.mean((softmax(z) - y0) ** 2)

def ce(z):
    return -log_softmax(z)[0]

print(numerical_gradient(mse, z0), numerical_gradient(ce, z0))
for name, f in [("MSE", mse), ("CE", ce)]:
    z, probs = z0.copy(), []
    for step in range(501):
        if step in [0, 5, 20, 100, 500]:
            probs.append(softmax(z)[0])            # 正確答案的機率
        z = z - 1.0 * numerical_gradient(f, z)     # 同樣的步伐（怎麼選是 EP08）
    print(name, np.round(probs, 4))

均方誤差的梯度只有 0.0013，交叉熵是 0.999，差約 750 倍：交叉熵 5 步就把正確答案推到 76%，均方誤差 100 步還卡在 0.15%。梯度＝（損失對 $q$ 的梯度）× Jacobian，而 Jacobian 每一格都很小（最大的是 $0.998 \times 0.002 \approx 0.002$），正確分數那一欄更是每一格都帶著 $q_{\text{正確}} = 0.001$。均方誤差對 $q$ 的梯度 $\frac{2}{3}(q - y)$ 不大，乘上去就幾乎是 0；交叉熵的 $-\frac{1}{q_{\text{正確}}} = -1000$ 剛好抵掉它，留下 $q - y$。**越有把握地答錯，均方誤差越推不動**。它不是永遠不動：正確答案的機率一點一點變大，Jacobian 跟著變大，到第 300 多步才突然衝上去。

把每一步都記下來畫成圖：左邊是正確分數那一格一開始的梯度，右邊是正確答案的機率隨步數的變化（橫軸是對數刻度）。印出來的是均方誤差、交叉熵第一次超過 0.5 的步數：

In [ ]:
g_ce, g_mse = numerical_gradient(ce, z0)[0], numerical_gradient(mse, z0)[0]   # 正確分數那一格的梯度
traj = {}
for name, f in [("MSE", mse), ("CE", ce)]:                 # 同樣的 500 步，這次每一步都記下來
    z, probs = z0.copy(), []
    for step in range(501):
        probs.append(softmax(z)[0])
        z = z - 1.0 * numerical_gradient(f, z)
    traj[name] = np.array(probs)
print(int(np.argmax(traj["MSE"] >= 0.5)), int(np.argmax(traj["CE"] >= 0.5)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1, 1.7]})
vals = [g_ce, g_mse]
ax1.bar([0, 1], vals, width=0.5, color=[BLUE, RED], edgecolor="white", linewidth=2)
for xi, v in zip([0, 1], vals):
    text = f"{v:.4f}" if v < -0.1 else f"{v:.4f}\n(almost no push)"
    ax1.text(xi, v - 0.03 if v < -0.1 else -0.04, text, ha="center", va="top", fontsize=10, color=INK)
ax1.axhline(0, color=MUTED, linewidth=1)
ax1.set_xticks([0, 1], ["cross-entropy", "mean squared error"])
ax1.set(ylim=(-1.15, 0.2), ylabel="gradient on the correct score", title="start: correct answer gets 0.001")
tidy(ax1, axis="y")

steps = np.arange(501)
ax2.plot(steps, traj["CE"], color=BLUE, linewidth=2, label="cross-entropy")
ax2.plot(steps, traj["MSE"], color=RED, linewidth=2, label="mean squared error")
for name, s, off in [("CE", 5, (8, -14)), ("MSE", 100, (-60, 10)), ("MSE", 500, (-140, -120))]:
    v = traj[name][s]
    ax2.plot([s], [v], "o", color=BLUE if name == "CE" else RED, markersize=8)
    arrow = {"arrowstyle": "-", "color": MUTED, "linewidth": 0.8} if s == 500 else None
    ax2.annotate(f"step {s}: {v:.4f}", (s, v), xytext=off, textcoords="offset points", fontsize=10, color=INK,
                 arrowprops=arrow)
ax2.set_xscale("symlog", linthresh=1)
ax2.set(xlim=(0, 520), ylim=(-0.03, 1.05), xlabel="step (log scale)",
        ylabel="probability of the correct answer", title="same step size, 500 steps")
ax2.legend(loc="center left", frameon=False)
tidy(ax2)
plt.show()

交叉熵第 4 步就超過一半、5 步到 0.76；均方誤差 100 步還在 0.0015，到第 325 步才超過一半，500 步到 0.96。

### 弄壞它 3：KL 的順序放反

資料是公平硬幣、模型以為偏成 0.9。第一行是你在第 8 節寫的 `kl_divergence` 兩個順序都算（nat）；第二行是 PyTorch 的 `F.kl_div` 照文件傳、放反、忘了取 log：

In [ ]:
try:
    print(kl_divergence(fair, biased), kl_divergence(biased, fair))   # 你在第 8 節寫的 kl_divergence
except NotImplementedError as e:
    print(f"{e}：先完成第 8 節的動手做，或執行最後的參考解答，再回來執行這格")
try:
    import torch
    import torch.nn.functional as F

    pt, qt = torch.tensor(fair), torch.tensor(biased)
    print(F.kl_div(torch.log(qt), pt, reduction="sum").item(),   # 照文件：先 log q（模型）、再 p（資料）
          F.kl_div(torch.log(pt), qt, reduction="sum").item(),   # 放反
          F.kl_div(qt, pt, reduction="sum").item())              # 忘了取 log
except ImportError:
    print("沒有安裝 PyTorch：跳過 F.kl_div 的部分（結果見下面的說明）")

第一行是 `0.5108256237659905 0.36806420716849714`，兩個順序差 0.14 nat；第二行是 `0.5108256237659905 0.368064207168497 -1.1931471805599454`。`F.kl_div(input, target)` 算的是 $\sum \text{target} \times (\log \text{target} - \text{input})$：`input` 要傳**模型的 log 機率**，`target` 是資料 $p$，結果才是 $D_{\mathrm{KL}}(p \,\|\, q)$。放反就變成 $D_{\mathrm{KL}}(q \,\|\, p)$；忘了取 log 算出 −1.19，KL 散度不可能是負的，程式卻不會報錯。

### 弄壞它 4：bit 和 nat 混用

工具五示範句的損失 1.750 是 nat，有人用 2 的次方算困惑度。依序印出用 $e$ 的次方、直接用 2 的次方，以及先換成 bit 再用 2 的次方：

In [ ]:
print(np.exp(loss_demo), 2 ** loss_demo, 2 ** (loss_demo / np.log(2)))

1.750 是用 $\ln$ 算的，困惑度要用 $e$ 的次方；拿 2 的次方只得到 3.36。要用 2，先把 nat 換成 bit（除以 $\ln 2$，得 2.525 bit），結果就一樣。看到損失或困惑度，先問單位：PyTorch 是 nat。

### 弄壞它 5：熵裡的 0 × log 0 沒處理

三格的分佈，其中一格是 0。第一行用工具二 `H_bits` 的寫法；第二行是 `np.log2(0.0)` 和 `0.0 * -np.log2(0.0)`；第三行是你在第 8 節寫的 `entropy`（前兩行會跳出 `RuntimeWarning`：那只是警告，不會讓 notebook 停下來）：

In [ ]:
p0 = np.array([0.5, 0.5, 0.0])
print(np.sum(p0 * -np.log2(p0)))           # 工具二 H_bits 的寫法：會跳出 RuntimeWarning
print(np.log2(0.0), 0.0 * -np.log2(0.0))   # 也會跳出 RuntimeWarning
try:
    print(entropy(p0, base=2))             # 你在第 8 節寫的 entropy
except NotImplementedError as e:
    print(f"{e}：先完成第 8 節的動手做，或執行最後的參考解答，再回來執行這格")

`nan`、`-inf nan`、`1.0`。前兩行各跳出兩個 `RuntimeWarning`：`divide by zero encountered in log2` 和 `invalid value encountered in multiply`（第二行是 `scalar multiply`）。`np.log2(0.0)` 是 `-inf`，`0.0 * inf` 是 `nan`（和 EP01 的 `inf / inf` 一樣），加任何數都是 `nan`。數學上這一項是 0，所以 `entropy` 要先跳過 $p = 0$ 的項，答案是 1 bit：多一個不可能的選項，猜謎不會變難。

## 下一集

這集知道了 LLM 的損失是平均交叉熵，也算出了它對分數的梯度 $q - y$，用固定的步伐走過幾步。下一集 **EP08 梯度下降與邏輯迴歸：第一個會學習的模型** 正式學沿著負梯度一步一步走：每一步走多遠由學習率決定，走得太大或太小會怎樣，也會親眼看到。

## 參考解答

先自己寫，卡住再看。執行下一格會用參考解答取代第 8 節的 6 個函式，並重新跑一次檢查；再下一格是 `bernoulli_entropy_bits`、`draw_bernoulli`、`draw_ce_stack` 的參考解答，會畫出本集重點那張熵與交叉熵的圖。之後可以回到上面，重新執行「和 PyTorch 對照」和弄壞它 3、5 那幾格。

想存成 `llm-labs/ep07/info.py` 的話，和前幾集的參考解答一樣：最上面加 `import numpy as np`（這集還要加 `softmax`、`log_softmax` 的兩行 import，寫在格子開頭的註解裡），刪掉最後一行 `check_info()`。畫圖的參考解答存成 `ep07/plot_entropy.py` 時，最上面加 `from pathlib import Path`、`import numpy as np` 和 `OUT = Path(__file__).parent / "entropy.png"`，最後一行 `show_entropy_plot()` 換成網站上那段 `if __name__ == "__main__":` 的主程式（一列兩張圖，存成 `OUT`）。

In [ ]:
# 只用 NumPy、基本的 Python 和前幾集自己寫的函式（softmax、log_softmax 已經定義在最上面）。
# 存成 llm-labs/ep07/info.py 時，最上面加這三行，並刪掉最後一行 check_info()：
#   import numpy as np
#   from ep01.softmax import softmax           # EP01 的穩定 Softmax
#   from ep02.mathlang import log_softmax      # EP02 寫過：z − logsumexp(z)，不先算 Softmax


def _to_unit(nats, base):
    """nat 換成以 base 為底的單位（base=2 是 bit）；base=None 就是 nat。"""
    if base is None:
        return nats
    return nats / np.log(base)


def entropy(p, base=None):
    """熵 H(p) = Σ p (−log p)，p 是一維的機率分佈。base=None 用 ln（nat），base=2 用 log₂（bit）。
    0 × log 0 當成 0：不能得到 nan，也不能跳出 RuntimeWarning。回傳 float。"""
    p = np.asarray(p, dtype=float)
    m = p > 0                                   # 只算 p > 0 的項：0 × log 0 = 0
    h = -np.sum(p[m] * np.log(p[m]))
    return float(_to_unit(h, base)) + 0.0       # + 0.0：把 −0.0 變成 0.0


def cross_entropy(p, q, base=None):
    """交叉熵 H(p, q) = Σ p (−log q)（p 是資料、q 是模型，同長度）。p = 0 的項當成 0；
    有一項 p > 0 而 q = 0，回傳 np.inf，不能跳出 RuntimeWarning。"""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    m = p > 0
    if np.any(q[m] == 0):
        return np.inf
    h = -np.sum(p[m] * np.log(q[m]))
    return float(_to_unit(h, base)) + 0.0


def kl_divergence(p, q, base=None):
    """KL 散度 D(p ‖ q) = Σ p log(p / q)。0 的處理和 cross_entropy 一樣。"""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    m = p > 0
    if np.any(q[m] == 0):
        return np.inf
    d = np.sum(p[m] * (np.log(p[m]) - np.log(q[m])))
    return float(_to_unit(d, base)) + 0.0


def _check_targets(logits, targets):
    """檢查 logits 是 (N, K)、targets 是長度 N、0 到 K − 1 的整數；不對就 raise ValueError。"""
    logits = np.asarray(logits, dtype=float)
    targets = np.asarray(targets)
    if logits.ndim != 2:
        raise ValueError(f"logits 要是 (N, K)，收到 {logits.shape}")
    n, k = logits.shape
    if targets.shape != (n,):
        raise ValueError(f"targets 要是長度 {n} 的一維陣列，收到 {targets.shape}")
    if not np.issubdtype(targets.dtype, np.integer):
        raise ValueError(f"targets 要是整數，收到 {targets.dtype}")
    if np.any(targets < 0) or np.any(targets >= k):
        raise ValueError(f"targets 要在 0 到 {k - 1} 之間：{targets}")
    return logits, targets


def cross_entropy_loss(logits, targets):
    """logits：(N, K) 的分數；targets：長度 N 的整數（每個位置正確 token 的 ID）。回傳平均交叉熵
    （nat，float），和 PyTorch 的 F.cross_entropy 一樣。logits 不是二維、targets 長度不對、
    不是整數、或不在 0 到 K − 1，raise ValueError。"""
    logits, targets = _check_targets(logits, targets)
    n = logits.shape[0]
    return float(-np.mean(log_softmax(logits, axis=-1)[np.arange(n), targets]))


def softmax_ce_grad(logits, targets):
    """cross_entropy_loss 對 logits 的梯度 (softmax(logits) − one-hot(targets)) / N，形狀 (N, K)。檢查同上。"""
    logits, targets = _check_targets(logits, targets)
    n, k = logits.shape
    return (softmax(logits, axis=-1) - np.eye(k)[targets]) / n


def perplexity(loss):
    """困惑度 exp(loss)，loss 是平均交叉熵（nat）。"""
    return float(np.exp(loss))


check_info()

In [ ]:
DATA = np.array([0, 1, 0, 2, 0, 2, 2, 0, 1, 0])     # EP06 的 10 個 token：冷 5、熱 2、好 3
STEPS = [0, 1, 2, 5, 10, 20]                        # 右圖要畫的步數
BLUE, RED, INK = "#2B7FD6", "#E04E3E", "#1E2A44"    # 熵、KL、文字


def bernoulli_entropy_bits(p):
    """白努利分佈的熵 H(p) = −p log₂ p − (1 − p) log₂(1 − p)（bit），p 可以是陣列；
    0 × log 0 當成 0，不能跳出警告。"""
    p = np.asarray(p, dtype=float)
    h = np.zeros_like(p)
    for prob in [p, 1 - p]:
        m = prob > 0
        h[m] -= prob[m] * np.log2(prob[m])
    return h


def softmax_1d(z):                                  # 這個檔案自己的 Softmax（不取名 softmax，才不會蓋掉最上面 EP01 的）
    e = np.exp(z - z.max())
    return e / e.sum()


def toy_training(steps):
    """EP06 的玩具模型：z 從 0 開始，每一步 z ← z − 1.0 × (q − p̂)（梯度 q − p̂，見「兌現 EP06」那一段）。
    回傳每個要畫的步數的 (交叉熵, KL)，單位 nat；熵 H(p̂) 另外回傳。"""
    p_hat = np.bincount(DATA, minlength=3) / DATA.size
    h = -np.sum(p_hat * np.log(p_hat))
    z, out = np.zeros(3), {}
    for step in range(max(steps) + 1):
        q = softmax_1d(z)
        if step in steps:
            ce = -np.sum(p_hat * np.log(q))
            out[step] = (ce, ce - h)
        z = z - 1.0 * (q - p_hat)
    return h, out


def draw_bernoulli(ax):
    """左圖：p 從 0 到 1（501 個點）畫 bernoulli_entropy_bits；標出 p = 0.5 和 p = 0.9 的點和數字。"""
    ps = np.linspace(0, 1, 501)
    ax.plot(ps, bernoulli_entropy_bits(ps), color=BLUE, linewidth=2)
    for p, text, offset in [(0.5, "fair coin: 1 bit", (-45, 10)), (0.9, "p = 0.9: 0.469 bit", (-125, -4))]:
        h = bernoulli_entropy_bits(np.array([p]))[0]
        ax.plot([p], [h], "o", color=RED, markersize=8)
        ax.annotate(text, (p, h), xytext=offset, textcoords="offset points", color=INK)
    ax.set(xlim=(0, 1), ylim=(0, 1.15), xlabel="p (probability of 1)", ylabel="entropy H (bit)",
           title="entropy of a coin: max at p = 0.5")
    ax.grid(True, color="0.9")


def draw_ce_stack(ax):
    """右圖：玩具模型的 z 從 0 開始，每一步換成 z − 1.0 × (q − p̂)；在 STEPS 那幾步畫堆疊長條（nat）：
    下段是熵 H(p̂)、上段是 KL，頂端寫交叉熵。"""
    h, out = toy_training(STEPS)
    x = np.arange(len(STEPS))
    kls = np.array([out[s][1] for s in STEPS])
    ax.bar(x, [h] * len(STEPS), width=0.6, color=BLUE, edgecolor="white", linewidth=2,
           label=f"entropy H(p) = {h:.4f} (stays)")
    ax.bar(x, kls, bottom=h, width=0.6, color=RED, edgecolor="white", linewidth=2,
           label="KL(p || q) (shrinks to 0)")
    for xi, s in zip(x, STEPS):
        ce, kl = out[s]
        ax.text(xi, ce + 0.02, f"{ce:.4f}", ha="center", va="bottom", fontsize=9, color=INK)
    ax.set_xticks(x, [str(s) for s in STEPS])
    ax.set(ylim=(0, 1.45), xlabel="training step (EP06 toy model)", ylabel="cross-entropy H(p, q) (nat)",
           title="cross-entropy = entropy + KL")
    ax.grid(True, axis="y", color="0.9")
    ax.set_axisbelow(True)
    ax.legend(loc="upper right")


show_entropy_plot()